# Decision Tree: Grid Search and RMSE

**Author:** Bakai

Model: `DecisionTreeRegressor`. Target: `price`. Metric: RMSE.

Steps: (1) reuse the shared setup from Dariya's preprocessing notebook, (2) baseline tree, (3) Grid Search, (4) overfitting check, (5) save results for the team table.

## 1. Shared setup (same as `01_data_preprocessing.ipynb`)

Same data, same `train_test_split(test_size=0.2, random_state=42)` and same `preprocessor`, so RMSE values are comparable with the other models.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

train_df = pd.read_csv("../data/challenge1_train.csv")

X = train_df.drop(columns=["price", "ID"])
y = train_df["price"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = ["bathrooms", "bedrooms", "square_feet", "latitude", "longitude"]
categorical_features = ["category", "has_photo", "state"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

print("X_train:", X_train.shape, "| X_val:", X_val.shape)

X_train: (5918, 8) | X_val: (1480, 8)


In [2]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# 5-fold CV on the training part only; the validation part is used for final reporting
cv = KFold(n_splits=5, shuffle=True, random_state=42)

## 2. Baseline tree (no limits)

A tree with no depth limit usually memorizes the training data, so train RMSE is tiny and CV RMSE is much larger. That gap is overfitting.

In [3]:
baseline = Pipeline([
    ("prep", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42)),
])

cvres = cross_validate(
    baseline, X_train, y_train, cv=cv,
    scoring="neg_root_mean_squared_error", return_train_score=True,
)
baseline.fit(X_train, y_train)

base_train = rmse(y_train, baseline.predict(X_train))
base_cv = -cvres["test_score"].mean()
base_val = rmse(y_val, baseline.predict(X_val))
print(f"Baseline  train RMSE: {base_train:.2f} | CV RMSE: {base_cv:.2f} | val RMSE: {base_val:.2f}")
print("Tree depth:", baseline.named_steps["model"].get_depth(),
      "| leaves:", baseline.named_steps["model"].get_n_leaves())

Baseline  train RMSE: 39.35 | CV RMSE: 559.81 | val RMSE: 544.24
Tree depth: 43 | leaves: 5443


## 3. Grid Search

Tuned hyperparameters (all control tree complexity):
- `max_depth`: maximum depth of the tree
- `min_samples_leaf`: minimum samples in a leaf
- `min_samples_split`: minimum samples needed to split a node
- `ccp_alpha`: cost-complexity pruning strength

Scoring is `neg_root_mean_squared_error`, so the best model has the highest (closest to 0) score.

In [4]:
pipe = Pipeline([
    ("prep", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42)),
])

param_grid = {
    "model__max_depth": [3, 5, 7, 10, 15, None],
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__min_samples_split": [2, 10, 20],
    "model__ccp_alpha": [0.0, 50.0, 500.0, 2000.0, 5000.0, 20000.0],
}

grid = GridSearchCV(
    pipe, param_grid, cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1, return_train_score=True,
)
grid.fit(X_train, y_train)

best_params = {k.replace("model__", ""): v for k, v in grid.best_params_.items()}
print("Best params:", best_params)
print(f"Best CV RMSE: {-grid.best_score_:.2f}")

Best params: {'ccp_alpha': 50.0, 'max_depth': 15, 'min_samples_leaf': 5, 'min_samples_split': 2}
Best CV RMSE: 482.37


In [5]:
res = pd.DataFrame(grid.cv_results_)
res["cv_rmse"] = -res["mean_test_score"]
res["train_rmse"] = -res["mean_train_score"]
cols = [c for c in res.columns if c.startswith("param_")] + ["train_rmse", "cv_rmse"]
res[cols].sort_values("cv_rmse").head(10).round(2)

,param_model__ccp_alpha,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,train_rmse,cv_rmse
124,50.0,15,5,10,339.50,482.37
123,50.0,15,5,2,339.50,482.37
51,0.0,15,5,2,333.42,483.89
52,0.0,15,5,10,333.42,483.89
125,50.0,15,5,20,370.19,484.09
53,0.0,15,5,20,367.83,484.45
136,50.0,None,5,10,332.87,485.08
135,50.0,None,5,2,332.87,485.08
137,50.0,None,5,20,365.32,486.19
65,0.0,None,5,20,360.78,487.02


## 4. Overfitting check: train vs validation RMSE

Compare the baseline and the tuned tree. The validation set (20% held out by Dariya's split) was not used for tuning.

In [6]:
best = grid.best_estimator_

tuned_train = rmse(y_train, best.predict(X_train))
tuned_cv = -grid.best_score_
tuned_val = rmse(y_val, best.predict(X_val))

summary = pd.DataFrame({
    "model": ["Decision Tree (baseline)", "Decision Tree (tuned)"],
    "train_rmse": [base_train, tuned_train],
    "cv_rmse": [base_cv, tuned_cv],
    "val_rmse": [base_val, tuned_val],
}).round(2)
summary["train_val_gap"] = (summary["val_rmse"] - summary["train_rmse"]).round(2)
summary

,model,train_rmse,cv_rmse,val_rmse,train_val_gap
0,Decision Tree (baseline),39.35,559.81,544.24,504.89
1,Decision Tree (tuned),346.44,482.37,503.72,157.28


In [7]:
tree = best.named_steps["model"]
print("Tuned tree depth:", tree.get_depth(), "| leaves:", tree.get_n_leaves())

feat_names = best.named_steps["prep"].get_feature_names_out()
imp = pd.Series(tree.feature_importances_, index=feat_names).sort_values(ascending=False)
imp.head(10).round(3)

Tuned tree depth: 15 | leaves: 356


num__square_feet    0.370
cat__state_CA       0.250
num__longitude      0.234
num__latitude       0.093
cat__state_IL       0.012
num__bedrooms       0.011
cat__state_MA       0.006
cat__state_DC       0.005
num__bathrooms      0.005
cat__state_CO       0.003
dtype: float64

## 5. Conclusion

- **Baseline (no limits):** depth 43 and 5,443 leaves. Train RMSE is 39.35 but CV RMSE is 559.81 and validation RMSE is 544.24, a gap of about 505. The tree memorizes the training data, which is strong overfitting.
- **Tuned tree** (`max_depth=15`, `min_samples_leaf=5`, `min_samples_split=2`, `ccp_alpha=50`): 356 leaves. Train RMSE 346.44, CV RMSE 482.37, validation RMSE 503.72. The train/validation gap fell from about 505 to about 157, and validation RMSE improved by about 40.5 (544.24 to 503.72). Some overfitting remains, since the gap is still large.
- **Most important features:** `square_feet` (0.37), `state_CA` (0.25), `longitude` (0.23) and `latitude` (0.09). Size and location drive the price predictions.
- **Limitations:** a single tree has high variance and is unstable, so Random Forest and Gradient Boosting are expected to reach a lower RMSE. The tuned tree is still useful as a simple, interpretable baseline.


## 6. Save results for the team comparison table

In [8]:
out = pd.DataFrame([{
    "model": "Decision Tree",
    "best_params": str(best_params),
    "train_rmse": round(tuned_train, 2),
    "cv_rmse": round(tuned_cv, 2),
    "val_rmse": round(tuned_val, 2),
}])

Path("../results").mkdir(exist_ok=True)
out.to_csv("../results/decision_tree.csv", index=False)
out

,model,best_params,train_rmse,cv_rmse,val_rmse
0,Decision Tree,"{'ccp_alpha': 50.0, 'max_depth': 15, 'min_samp...",346.44,482.37,503.72
